In [ ]:
# uv add pymilvus
# uv add pymilvus[milvus-lite]  单文件模式 windows不支持

In [1]:
import numpy as np
from pymilvus import  connections,Collection,FieldSchema,CollectionSchema,DataType,utility

In [2]:
items = [
    {"type": "phone", "id": "用户A", "number": 13800001234},
    {"type": "phone", "id": "用户B", "number": 13800005678},
    {"type": "order", "id": "订单1001", "number": 203011010001},
    {"type": "order", "id": "订单1002", "number": 203011010123},
    {"type": "order", "id": "订单2001", "number": 203012150045},
    {"type": "phone", "id": "用户C", "number": 13912345678},
    {"type": "phone", "id": "用户D", "number": 13798765432},
    {"type": "order", "id": "订单3001", "number": 205001020333},
    {"type": "order", "id": "订单3002", "number": 205001020777},
    {"type": "phone", "id": "用户E", "number": 13622223333},
]

numbers = np.array([row['number'] for row in items ],dtype='float32')

In [7]:
# MilvusException: <MilvusException: (code=65535, message=invalid dimension: 1. should be in range 2 ~ 32768)>
# 数据库要求维度，不能少于2
dimension = 2
# vectors = numbers.reshape(-1,1)
vectors = np.stack([numbers, numbers],axis=1)
vectors

array([[1.3800002e+10, 1.3800002e+10],
       [1.3800006e+10, 1.3800006e+10],
       [2.0301101e+11, 2.0301101e+11],
       [2.0301101e+11, 2.0301101e+11],
       [2.0301215e+11, 2.0301215e+11],
       [1.3912346e+10, 1.3912346e+10],
       [1.3798766e+10, 1.3798766e+10],
       [2.0500102e+11, 2.0500102e+11],
       [2.0500102e+11, 2.0500102e+11],
       [1.3622223e+10, 1.3622223e+10]], dtype=float32)

In [8]:
# 链接数据库
connections.connect(
    host='localhost',
    port ='19530',
    alias = 'default'
)

# 定义集合 Schema
fields = [
    FieldSchema(name='id',dtype=DataType.INT64,is_primary=True,auto_id=True),
    FieldSchema(name='type',dtype=DataType.VARCHAR,max_length = 50),
    FieldSchema(name='item_id',dtype=DataType.VARCHAR,max_length = 100),
    FieldSchema(name='number',dtype=DataType.INT64),
    FieldSchema(name ='embedding',dtype=DataType.FLOAT_VECTOR,dim=dimension)
]
schema = CollectionSchema(fields,'数字向量集合')  # 创建集合Schema对象

collection_name = 'numbers_collection'       # 集合名称
if utility.has_collection(collection_name):  # 判断有没有这个集合
    utility.drop_collection(collection_name) # 删除集合

collection = Collection(collection_name,schema)

In [ ]:
# 准备数据
types = [item.get('type') for item in items]
item_ids = [item.get('id') for item in items]
embeddings = vectors.tolist() # 转换成python的list

data = [
    types,
    item_ids,
    numbers.astype('int64'),
    embeddings
]
# 将数据存放到集合中
collection.insert(data)

(insert count: 10, delete count: 0, upsert count: 0, timestamp: 462417171056689171, success count: 10, err count: 0

In [ ]:
# 创建索引,类似FAISS中 IndexFlatL2
index_params ={
    'metric_type':'L2',  # 欧式距离
    'index_type':'FLAT'  # 精确搜索
}
collection.create_index('embedding',index_params)  # 创建索引

# 加载到内存
collection.load()

print(f'已向索引添加 {len(items)} 个数字向量 (维度={dimension})')

已向索引添加 10 个数字向量 (维度=2)


In [12]:
# 查询的数据
query_number = 205001020500
query_vec = np.array([[query_number,query_number]],dtype='float32')

# 获取相近多个数据
k = 5
# 获取数据
results = collection.search(
    data = query_vec.tolist(),   # 查询的数据
    anns_field='embedding',      # 索引的字段
    param = {'metric_type':'L2'}, # 搜索的参数
    limit=k,
    output_fields= ['type','item_id','number']
)
print(f'查询数字:{query_number}')
print(f'Top-{k} 最相近的数字(L2距离，越小表示越相似)：')

for hits in results:
    for rank,hit in enumerate(hits,start=1):
        row = {
            "type":hit.entity.get('type'),
            "item_id":hit.entity.get('item_id'),
            "number":hit.entity.get('number')
        }
        print(f'第{rank}个最相似的数字是： 距离 {hit.distance:.0f} ： 类型={row["type"]}，编号={row["item_id"]}，数字={row["number"]}')

查询数字:205001020500
Top-5 最相近的数字(L2距离，越小表示越相似)：
第1个最相似的数字是： 距离 0 ： 类型=order，编号=订单3001，数字=205001015296
第2个最相似的数字是： 距离 0 ： 类型=order，编号=订单3002，数字=205001015296
第3个最相似的数字是： 距离 7911208812952944640 ： 类型=order，编号=订单2001，数字=203012145152
第4个最相似的数字是： 距离 7920205017091407872 ： 类型=order，编号=订单1001，数字=203011014656
第5个最相似的数字是： 距离 7920205017091407872 ： 类型=order，编号=订单1002，数字=203011014656
